#### 1. Data Loading & Date Parsing

-imports

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os


-Load csv 

In [30]:
df = pd.read_csv("../data/nigeria.csv")


-Add country identity

In [31]:
df["Country"] = "Nigeria"

-Convert YEAR and DOY to Datetime

In [32]:
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')

-Extract Month for Seasonal Analysis

In [33]:
df['Month'] = df['Date'].dt.month

-Reordering columns for better readability


In [34]:
cols = ['Date', 'Country', 'Month', 'YEAR', 'DOY', 'T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR']
df= df[cols + [c for c in df.columns if c not in cols]]

print("First 5 rows of processed Nigeria data:")
df.head()

First 5 rows of processed Nigeria data:


,Date,Country,Month,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,T2M_RANGE,RH2M,WS2M,WS2M_MAX,PS,QV2M
0,2015-01-01,Nigeria,1,2015,1,25.23,29.25,22.06,0.0,7.19,68.26,1.73,2.61,100.86,13.36
1,2015-01-02,Nigeria,1,2015,2,26.16,29.41,22.87,0.0,6.54,73.23,1.42,1.95,100.94,15.37
2,2015-01-03,Nigeria,1,2015,3,25.66,29.02,22.63,0.0,6.39,78.71,1.69,2.33,101.06,15.98
3,2015-01-04,Nigeria,1,2015,4,24.11,27.27,19.92,0.0,7.35,63.66,2.15,3.80,101.09,11.65
4,2015-01-05,Nigeria,1,2015,5,23.40,27.28,18.18,0.0,9.10,59.45,1.88,3.48,101.03,10.40


##### Analytical Reasoning
Analytical Note: Converting orbital data to a temporal format is essential for mapping the West African Monsoon. This allows us to track the "Little Dry Season" (August Break) in the south. Monitoring these shifts is vital for Nigeria’s COP32 position, as changes in monsoon intensity are directly linked to catastrophic urban flooding in Lagos and the Niger Delta.

#### 2. Summary Statistics & Missing-Value 


-Replace NASA sentinel values
* NASA POWER uses -999 as a sentinel value for missing data;
these were replaced with NaN to prevent statistical bias.


In [51]:
df.replace(-999, np.nan, inplace=True)

,Date,Country,Month,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,T2M_RANGE,RH2M,WS2M,WS2M_MAX,PS,QV2M
0,2015-01-01,Nigeria,1,2015,1,25.23,29.25,22.06,0.00,7.19,68.26,1.73,2.61,100.86,13.36
1,2015-01-02,Nigeria,1,2015,2,26.16,29.41,22.87,0.00,6.54,73.23,1.42,1.95,100.94,15.37
2,2015-01-03,Nigeria,1,2015,3,25.66,29.02,22.63,0.00,6.39,78.71,1.69,2.33,101.06,15.98
3,2015-01-04,Nigeria,1,2015,4,24.11,27.27,19.92,0.00,7.35,63.66,2.15,3.80,101.09,11.65
4,2015-01-05,Nigeria,1,2015,5,23.40,27.28,18.18,0.00,9.10,59.45,1.88,3.48,101.03,10.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4103,2026-03-27,Nigeria,3,2026,86,29.05,32.43,26.69,2.82,5.74,76.60,2.35,3.39,100.58,19.10
4104,2026-03-28,Nigeria,3,2026,87,28.72,31.98,27.14,5.19,4.84,79.61,2.55,3.17,100.64,19.49
4105,2026-03-29,Nigeria,3,2026,88,27.72,29.53,26.21,1.43,3.32,82.83,1.10,1.78,100.61,19.22
4106,2026-03-30,Nigeria,3,2026,89,28.42,31.17,26.36,0.85,4.81,77.73,2.30,3.40,100.53,18.73


-Duplicate Check


In [52]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
df = df.drop_duplicates()

Duplicate rows found: 0


-Missing Value 


In [53]:
missing = df.isna().sum()
missing_percent = (missing/ len(df)) * 100

print("\nMissing Value Percentages per Column:")
print(missing_percent[missing_percent > 0])


Missing Value Percentages per Column:
Series([], dtype: float64)


-Generate Summary Statistics
* We focus on temperature and precipitation for climate trends


In [54]:
summary_stats= df[['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M']].describe()
summary_stats

,T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M
count,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000
mean,26.656928,28.914667,24.886461,4.213914,85.237040
std,1.123335,1.294345,1.396727,7.266742,5.446007
min,21.120000,25.260000,15.170000,0.000000,54.400000
25%,25.720000,27.920000,24.100000,0.330000,83.930000
50%,26.820000,28.990000,25.100000,1.840000,86.350000
75%,27.540000,29.910000,25.860000,5.200000,88.500000
max,29.290000,32.880000,27.790000,166.100000,93.790000


##### Statistical Interpretation

* Temperature: Nigeria’s mean T2M will likely stay consistently high (27-30°C). However, for Nigeria, we must look closely at RH2M (Humidity). High temperature combined with high humidity indicates lethal wet-bulb temperatures, a major health risk in urban centers like Lagos.

* Missing Data: If PRECTOTCORR has >5% missing values, it is highly problematic for flood modeling. In the tropical south, missing data often occurs during the most intense storms when sensor equipment might fail; this leads to a dangerous underestimation of flood risk in the Niger Delta.